In [492]:
import google.generativeai as genai
import os

genai.configure(api_key="<>")

## SCRIPTS FOR AI GENERATION ABSTRACT USING TOPICS

In [ ]:
import pandas as pd
import time

def file_to_df(file_name):
    if os.path.exists(file_name):
        df_write = pd.read_csv(file_name)
    else:
        columns = ['Author_name', 'topic', 'title', 'Abstract','year', 'Source', 'label']
        df_write = pd.DataFrame(columns=columns)
    return df_write

def data_range(df, start, end):
    topics = df["topic"].values[start: end]
    title = df["title"].values[start: end]

    return { "topics" : topics,
              "title" : title
                }

# generate_abstract(data_range["abstract"])

def generate_abstract(title):  # titles: list or string of 20 research topics
    model = genai.GenerativeModel("gemini-1.5-flash")
    
    response = model.generate_content(f'''
    You will be provided with a research title. Generate each abstract individually and return only abstract. 
                                      Use the separator "|||" and ensure no other text or commentary is added.
    Each abstract should be between 200 to 250 words
                                      
    titles:
    {title}
    
    Generated Abstracts:
    
    ''')
    abstracts = [abstract.strip() for abstract in response.text.split('|||') if abstract.strip()]
    return abstracts


def add_rows(df_write, data_range, generated_abstract_array):
    topics = data_range["topics"]
    abstracts = generated_abstract_array
    titles = data_range["title"]

    for topic, abstract, title in zip(topics, abstracts, titles):
        new_row = pd.DataFrame({
            'Author_name': "gemini",
            "topic": topic,
            'title': title,
            'Abstract': abstract,
            'year': 2024,
            'Source': 'gemini',
            'label': 'AI'
        }, index=[0])
        df_write = pd.concat([df_write, new_row], ignore_index=True)

    return df_write  # updated DataFrame

    

    

In [ ]:
# every batch will save
def main(df, end, start, batch=20):

    # starting index
    start = start
    # ending index
    end = end
    
    file_name = '../dataset/AI_GENERATED_ABSTRACT_USING_GEMINI.csv'
    df_write = file_to_df(file_name)
    print(f"Expected time needed for this code to be executed: {(end * 1) // 60} minutes")
    counter = 0

    for end_range in range(start, end, batch):
        if end_range + batch -1 < end:
            start_time = time.time()
            data_provided = data_range(df = df ,start = end_range, end = end_range + batch)
            data = generate_abstract(titles = data_provided["title"])
            df_write = add_rows(df_write, data_provided, data)  # Update df_write with each batch
            end_time = time.time()

            # update to CSV after each batch
            df_write.to_csv(file_name, index=False)
            counter += 1
            print(f"Time Taken {round(end_time - start_time, 2)} seconds")
            print(f"Loop completed till {end_range + batch} | request {counter}")
                    

            print(f"Data successfully written to {file_name}")


In [ ]:
df = pd.read_csv("../dataset/abstract_dataset.csv")
main(df, batch = 5, start= 14506, end = len(df) )

Expected time needed for this code to be executed: 242 minutes
Time Taken 16.19 seconds
Loop completed till 14539 | request 1
Data successfully written to AI_GENERATED_ABSTRACT_USING_GEMINI.csv


In [489]:
len(df)

14539

In [ ]:
# 150 abstracts will take 45000 tokens generate in 28*3

In [475]:
28*(len(df)//50)//60 # minutes 

135

# SCRIPT FOR GENERATING REPHRASED ABSTRACTS

In [ ]:
# def splitter(abstracts_vector):
#     return abstracts_vector.split("\n \n")

# def generate_abstract(data):
#     model = genai.GenerativeModel("gemini-1.5-flash")
#     response = model.generate_content(f'''
#     read the abstract and rephrase it
#     {data["Abstract"].values} 
#     just provide each rephrased abstract leaving a 2 line of gap, no other words only abstract

#     ''')

#     return response.text
# generate = generate_abstract(data)

# import pandas as pd

# def file_to_df(file_name):
#     if os.path.exists(file_name):
#         # Read the existing data into a DataFrame
#         df_write = pd.read_csv(file_name)
#     else:
#         # If the file doesn't exist, create a new DataFrame
#         columns = ['Author_name', 'topic', 'title', 'Abstract','year', 'Source', 'label']
#         df_write = pd.DataFrame(columns=columns)
        

# def add_rows(topics, titles, splitter_abstract):
#     for topic, abstract, title in zip(topics, splitter_abstract, titles):        
#         new_row = pd.DataFrame({
#                     'Author_name': "gemini",
#                     "topic" : topic,
#                     'title': title,
#                     'Abstract': abstract,
#                     'year': 2024,
#                     'Source': 'gemini',
#                     'label': 'AI'
#                 })
                
#         df_write = pd.concat([df_write, new_row], ignore_index=True)

#         df_write.to_csv(file_name, index=False)
#         print(f"Data successfully written to {file_name}")




In [344]:
import pandas as pd
import time

def file_to_df(file_name):
    if os.path.exists(file_name):
        # Read the existing data into a DataFrame
        df_write = pd.read_csv(file_name)
    else:
        # If the file doesn't exist, create a new DataFrame
        columns = ['Author_name', 'topic', 'title', 'Abstract','year', 'Source', 'label']
        df_write = pd.DataFrame(columns=columns)
    return df_write

def data_range(df, start, end):
    abstracts = df["Abstract"].values[start: end]
    topics = df["topic"].values[start: end]
    title = df["title"].values[start: end]

    return { "abstract":abstracts,
             "topics" : topics,
              "title" : title
                }

# generate_abstract(data_range["abstract"])

def generate_abstract(data):  # data: array of 20 abstracts
    model = genai.GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(f'''
    You will be provided with 20 research abstracts. Rephrase each abstract individually and improve the quality of english and return each rephrased version without additional text or formatting. Use the separator "|||", and ensure no other text or commentary is added.
    
    Abstracts:
    {data}
    
    Rephrased Abstracts:
    ''')

    # Split the response based on the custom separator "|||"
    return response.text.split('|||\n')

# Function to add rows of data to the DataFrame and save it to CSV

# # add_rows(df_write, file_name, data_range )
# def add_rows(df_write, file_name, data_range, generated_abstract_array):
#     topics =  data_range["topics"]
#     abstracts = generated_abstract_array
#     titles = data_range["title"]

#     for topic, abstract, title in zip(topics, abstracts, titles):
#         new_row = pd.DataFrame({
#             'Author_name': "gemini",
#             "topic": topic,
#             'title': title,
#             'Abstract': abstract,
#             'year': 2024,
#             'Source': 'gemini',
#             'label': 'AI'
#         }, index=[0])
#         file_name = 'rephrased_dataset.csv'
#         df_write = pd.concat([df_write, new_row], ignore_index=True)

#     df_write.to_csv(file_name, index=False)
#     print(f"Data successfully written to {file_name}", end = "  ")

# Adjusted add_rows function
def add_rows(df_write, data_range, generated_abstract_array):
    topics = data_range["topics"]
    abstracts = generated_abstract_array
    titles = data_range["title"]

    for topic, abstract, title in zip(topics, abstracts, titles):
        new_row = pd.DataFrame({
            'Author_name': "gemini",
            "topic": topic,
            'title': title,
            'Abstract': abstract,
            'year': 2024,
            'Source': 'gemini',
            'label': 'AI'
        }, index=[0])
        df_write = pd.concat([df_write, new_row], ignore_index=True)

    return df_write  # Return the updated DataFrame

    

    

In [ ]:
# every batch will save
def main(df, bulk=20):
    file_name = '../dataset/rephrased_dataset.csv'
    df_write = file_to_df(file_name)
    print(f"Expected time needed for this code to be executed: {((len(df)) * 1) // 60} minutes")
    counter = 1

    for end_range in range(14520, len(df), bulk):
        if end_range + bulk - 1 < len(df):
            start_time = time.time()
            data_provided = data_range(df, end_range, end_range + bulk)
            data = generate_abstract(data_provided["abstract"])
            df_write = add_rows(df_write, data_provided, data)  # Update df_write with each batch
            end_time = time.time()
            print(f"Time Taken {round(end_time - start_time, 2)} seconds")
            print(f"Loop completed till {end_range + bulk} | request {counter}")
            
            # update to CSV after each batch
            df_write.to_csv(file_name, index=False)
            counter += 1

    print(f"Data successfully written to {file_name}")


In [ ]:
df = pd.read_csv("../dataset/abstract_dataset.csv")
main(df, bulk=19)

Expected time needed for this code to be executed: 242 minutes
Time Taken 20.89 seconds
Loop completed till 14539 | request 1
Data successfully written to rephrased_dataset.csv


In [ ]:
'''

on request 1 :- 20 abstracts take 20 second 
           15:- 300 abstracts will take 300 sec
          714:- 14290 abstracts will take 238 mins
          

               20*1500
'''

In [380]:
14290//60

238

In [384]:
20*60

1200

In [385]:
150*20

3000

In [386]:
18*60

1080

## Approx time 

In [318]:
1 min 3 request --> 60
10 min 30 request --> 600
100 min 300 request --> 6000
200 min 600 request --> 12000

15000

14320

# SCRIPT FOR MISSING REPHRASED ABSTRACTS

In [ ]:
import pandas as pd

# Sample DataFrame with columns "titles" and "abstracts"
data_file  = pd.read_csv("../dataset/AI_GENERATED.csv")
dataframe = pd.DataFrame(data_file)

# Find rows where "abstracts" are empty and retrieve their titles and indices
empty_abstracts = dataframe[dataframe['Abstract'].isna()].copy()
empty_abstracts['index'] = empty_abstracts.index

print("Titles and indices of rows with empty abstracts:")
print(empty_abstracts)


Titles and indices of rows with empty abstracts:
Empty DataFrame
Columns: [Author_name, topic, title, Abstract, year, Source, label, index]
Index: []


In [376]:
dataframe['Abstract'].isna() if dataframe['Abstract'].isna().any() else print("")

In [2]:
import pandas as pd

In [ ]:
topic_df = pd.read_csv("../dataset/AI_GENERATED_ABSTRACT_USING_GEMINI_REPHRASED.csv")
paraphrase = pd.read_csv("AI_GENERATED_ABSTRACT_USING_GEMINI_TopicWise.csv"
                         )
len(topic_df)

14291

In [4]:
len(paraphrase)

13780

LLAMA 3.1 8B instant


requet per minute = 30
request per day   = 14000
each abstract --> 300 words
ideally 20000 tokens make --> 15000 words

15000 // 300  === 50 absract per min (ideal)


let's take 30 abstracts per min

1.5 mins --> 30 abstract   (30 * 300   ) --> 9000 words
15  mins --> 300 abstract  (300*300 words) -->90000

60 mins  -->  1200 abstract (360000 words) 478800.0 tokens

1200 abstracts per day





In [1]:
1200*300

360000

In [2]:
1.33*360000

478800.0